# F1 Strategist — Colab Smoke Notebook

Smoke-test version of the training pipeline. Runs in **~7 minutes on a Colab Free T4**.

**Owner:** Person 2 (Tanish).

**Goal:** demonstrate that the env + TRL stack work end-to-end on commodity hardware. The full 500-step run goes on the RTX 5090; this notebook is the W1 minimum requirement (Colab-runnable training script).

## What this notebook does

1. Install OpenEnv, TRL, and supporting deps (~3 min)
2. Pull the F1 Strategist env via `from_hf_space("Deltasthic/f1-strategist")` (Docker pulled into Colab)
3. Smoke train: 30 local environment-policy steps (~3 min) or switch `--backend trl` for GPU GRPO
4. Plot the reward curve
5. Print one full rollout transcript

## What this notebook does NOT do

- The full 500-step run (needs ≥32 GB VRAM, won't fit on T4)
- Final eval on held-out seeds (use `evaluate.py` on the GPU server)
- Postmortem ablation (separate notebook or use the GPU server)

## 1. Install dependencies

Pin to versions known-good with Qwen3 + TRL 0.14. See `TRAINING.md` for the full rationale.

In [ ]:
%pip install -q "openenv-core>=0.2.3"
%pip install -q "transformers>=4.55.4" "accelerate>=1.3" matplotlib pandas numpy
%pip install -q "trl==0.14.0" "peft>=0.13.2" "datasets>=3.2.0"
import pathlib, subprocess, sys

repo = pathlib.Path("f1-strategist")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/Deltasthicc/f1-strategist.git"], check=True
    )
sys.path.insert(0, str(repo.resolve()))

## 2. Connect to the F1 Strategist env

Two options:
- **From the published HF Space:** zero local setup
- **From a local Docker build:** if you cloned the repo into Colab

In [ ]:
%cd f1-strategist
%pip install -q -e .
from server.environment import F1StrategistEnvironment
from server.scenarios import SCENARIOS

env = F1StrategistEnvironment()
# Use options dict with a scenario object, not a task string
obs = env.reset(seed=0, options={"scenario": SCENARIOS["dry_strategy_sprint"]})
import json
print(json.dumps(obs.model_dump(), indent=2))

## 3. Smoke-train Qwen3-0.6B with GRPO

30 steps with the local smoke backend. On a GPU runtime, change `--backend local-smoke` to `--backend trl --allow-cpu` only for debugging; use the 5090 recipe for real training.

In [ ]:
# local-smoke backend: runs scripted-policy rollouts and records a reward curve.
# No GPU or LLM needed — perfect for verifying the pipeline before full GRPO.
!python train.py \
    --backend local-smoke \
    --task multi \
    --max-steps 30 \
    --batch-size 1 --grad-accum 1 \
    --output-dir ./grpo_colab_smoke \
    --logging-steps 5 \
    --save-steps 15

## 4. Plot the reward curve

In [ ]:
!python scripts/plot_training_curve.py --run-dir ./grpo_colab_smoke --output results/training_loss_curve.png
from IPython.display import Image, display

display(Image("results/training_loss_curve.png"))

## 5. Print one rollout transcript

Run the (just-trained) policy through one episode and print every (obs, action, reward) triple.

In [ ]:
from inference import run_inference

# "heuristic" uses the built-in keyword policy — no checkpoint needed.
# To use a trained checkpoint pass: model="./grpo_colab_smoke/checkpoint-15"
scores = run_inference(
    model="heuristic",
    task="dry_strategy_sprint",
    n_episodes=1,
    verbose=True,
)
print(f"\nEpisode scores: {scores}")

## What's next

If this notebook ran cleanly:
- The env is reachable from Colab
- The TRL stack imports without conflict
- The reward signal is non-zero (you should see the curve trend up over 30 steps)

For real numbers, run on a 32 GB+ GPU via [`TRAINING.md`](https://github.com/Deltasthicc/f1-strategist/blob/main/TRAINING.md):

```bash
python train.py --model Qwen/Qwen3-4B --max-steps 500 --batch-size 1 --grad-accum 32
```